# ⬡ RootSearch — GLM-4-9B-Chat Colab Server (Cloudflare Tunnel)
### خادم تشغيل نموذج الذكاء الاصطناعي GLM-4-9B-Chat بسرعة فائقة باستخدام كرت الشاشة (GPU) ونفق Cloudflare الآمن

هذا الملف مخصص للتشغيل على **Google Colab** لإنشاء خادم ويب عام وآمن باستخدام نفق **Cloudflare Tunnel (trycloudflare)** الموثوق بدون تسجيل وبدون الحاجة لـ Ngrok.

---

## 1️⃣ التحقق من مواصفات كرت الشاشة (GPU)
تأكد من أنك قمت بتغيير نوع وقت التشغيل إلى **GPU** (T4 GPU أو L4 أو A100).

In [ ]:
!nvidia-smi

## 2️⃣ تثبيت المكتبات والمتطلبات
نقوم بتثبيت نسخة مستقرة ومتوافقة بالكامل من مكتبة `transformers` لضمان عمل النموذج بدون أي أخطاء متعلقة بالأوزان المشتركة أو الخصائص المفقودة.

In [ ]:
# تثبيت نسخة مستقرة ومجربة متوافقة مع GLM-4
!pip install -q "transformers==4.41.2" accelerate bitsandbytes fastapi uvicorn nest_asyncio jinja2

## 3️⃣ تحميل وتجهيز نموذج GLM-4-9B-Chat بدقة 4-bit
نستخدم تقنية 4-bit Quantization لضغط النموذج وتوفير استهلاك الذاكرة العشوائية لكرت الشاشة (VRAM) مع زيادة هائلة في سرعة توليد النصوص.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig, BitsAndBytesConfig

model_id = "THUDM/glm-4-9b-chat"

print("⏳ جاري تحميل وتجهيز المُرَمّز والإعدادات...")

# إعدادات الضغط المتقدمة
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# تحميل المُرَمّز
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# تحميل التكوين وحل مشكلة max_length بشكل احتياطي
config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
config.max_length = getattr(config, "seq_length", 8192)

print("⏳ جاري تحميل وتوزيع النموذج على كرت الشاشة (قد يستغرق 3-5 دقائق)...")
# تحميل النموذج مع التوزيع الآلي على كرت الشاشة
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)

print("✅ تم تحميل نموذج GLM-4-9B-Chat بنجاح وهو جاهز للعمل مع كرت الشاشة بأقصى سرعة!")

## 4️⃣ إنشاء واجهة برمجية خفيفة (FastAPI Server)
سنقوم ببناء خادم ويب يدعم صيغة طلبات **OpenAI API** لتبسيط الاندماج مع محرك البحث.

In [ ]:
import nest_asyncio
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import uvicorn
import threading
import time
import json
import re
import traceback

app = FastAPI(title="RootSearch GLM-4 Colab Server")

@app.get("/")
def home():
    return {
        "status": "healthy",
        "model": "GLM-4-9B-Chat",
        "device": str(next(model.parameters()).device)
    }

@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    try:
        body = await request.json()
        messages = body.get("messages", [])
        temperature = body.get("temperature", 0.3)
        max_tokens = body.get("max_tokens", 1024)
        
        # صياغة النص باستخدام قالب المحادثة الرسمي
        prompt = tokenizer.apply_chat_template(
            messages, 
            add_generation_prompt=True, 
            tokenize=False
        )
        
        # تحويل النص إلى معرفات رقمية ونقلها إلى كرت الشاشة
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=True if temperature > 0 else False,
                temperature=temperature if temperature > 0 else 1.0,
                top_p=0.9,
                eos_token_id=tokenizer.eos_token_id
            )
        
        # فك تشفير المخرجات فقط
        input_len = inputs["input_ids"].shape[1]
        response_tokens = outputs[0][input_len:]
        response_text = tokenizer.decode(response_tokens, skip_special_tokens=True).strip()
        
        # إرجاع استجابة مطابقة لمعيار OpenAI
        return JSONResponse({
            "id": "chatcmpl-colab",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": "glm-4-9b-chat",
            "choices": [
                {
                    "index": 0,
                    "message": {
                        "role": "assistant",
                        "content": response_text
                    },
                    "finish_reason": "stop"
                }
            ]
        })
    except Exception as e:
        print("❌ حدث خطأ أثناء التوليد:")
        traceback.print_exc()
        return JSONResponse({"error": str(e)}, status_code=500)

## 5️⃣ فتح نفق عام وتشفير الخدمة باستخدام Cloudflare Tunnel
سنقوم بتحميل أداة `cloudflared` وتشغيلها لفتح نفق عام آمن مجاني بالكامل بدون تسجيل حساب وبسرعة هائلة.

In [ ]:
# تحميل أداة Cloudflare Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# تشغيل خادم FastAPI محلياً في خيط خلفي (Background Thread)
nest_asyncio.apply()
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)

t = threading.Thread(target=run_server)
t.daemon = True
t.start()
time.sleep(2) # انتظار خادم الويب ليصبح نشطاً

# إغلاق أي نفق قديم قيد التشغيل
!pkill cloudflared

import subprocess
import os

log_file = "cloudflared.log"
if os.path.exists(log_file):
    os.remove(log_file)

print("⏳ جاري تشغيل نفق Cloudflare والحصول على الرابط...")
with open(log_file, "w") as f:
    process = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
        stdout=f,
        stderr=f
    )

time.sleep(6) # انتظار حتى يولد كولدفلير الرابط العام

tunnel_url = None
with open(log_file, "r") as f:
    log_content = f.read()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
    if match:
        tunnel_url = match.group(0)

if tunnel_url:
    print("\n🔥 [نجح الاتصال] نفق Cloudflare نشط الآن!")
    print(f"🔗 الرابط العام الخاص بك هو: {tunnel_url}")
    print("\n👉 انسخ هذا الرابط وضعه في ملف الـ .env الخاص بمشروعك كـ GLM_API_URL")
else:
    print("❌ فشل فتح نفق Cloudflare. تفاصيل سجل الأخطاء:")
    with open(log_file, "r") as f:
        print(f.read())